In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split

In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

e:\Anaconda\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [3]:
img_width = (224, 224)
batch_size = 32

datagen = ImageDataGenerator(rescale=1./255)

train_data = datagen.flow_from_directory(
    'dataset/train/train',
    target_size = img_width,
    batch_size = batch_size,
    class_mode = "categorical"
)

val_data = datagen.flow_from_directory(
    'dataset/val',
    target_size = img_width,
    batch_size = batch_size,
    class_mode = "categorical"
)

Found 16854 images belonging to 33 classes.
Found 5824 images belonging to 33 classes.


In [4]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import models, layers
from tensorflow.keras.optimizers import Adam

In [5]:
base_model = MobileNetV2(
    input_shape = (224, 224, 3),
    include_top = False,
    weights = "imagenet"
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(train_data.num_classes, activation="softmax")
])

model.compile(
    optimizer = Adam(learning_rate=0.0001),
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)

model.fit(train_data, epochs=3, validation_data=val_data)

Epoch 1/3
527/527 ━━━━━━━━━━━━━━━━━━━━ 311s 584ms/step - accuracy: 0.5977 - loss: 1.6271 - val_accuracy: 0.9928 - val_loss: 0.2936
Epoch 2/3
527/527 ━━━━━━━━━━━━━━━━━━━━ 174s 331ms/step - accuracy: 0.9333 - loss: 0.3552 - val_accuracy: 0.9985 - val_loss: 0.0673
Epoch 3/3
527/527 ━━━━━━━━━━━━━━━━━━━━ 172s 326ms/step - accuracy: 0.9753 - loss: 0.1575 - val_accuracy: 0.9993 - val_loss: 0.0257


In [9]:
from tensorflow.keras.preprocessing import image

img = image.load_img("avocado.jpg", target_size=(224,224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)
print(prediction)

pred_class = np.argmax(prediction, axis=1)
print("Predicted class index:", pred_class[0])

class_names = list(train_data.class_indices.keys())
print("Predicted class:", class_names[pred_class[0]])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
[[0.00100195 0.01981958 0.01317383 0.01039236 0.00441152 0.00150555
  0.0044669  0.06655654 0.00240481 0.02774722 0.04477156 0.07374986
  0.01122124 0.00613447 0.07881177 0.00306317 0.00051047 0.00488905
  0.00244349 0.0119146  0.01786828 0.00293656 0.41171542 0.01848743
  0.02047979 0.05689288 0.00157075 0.01337273 0.00793169 0.00590534
  0.0047099  0.04467741 0.00446186]]
Predicted class index: 22
Predicted class: Pear


In [7]:
print(train_data.class_indices)

{'Apple Braeburn': 0, 'Apple Granny Smith': 1, 'Apricot': 2, 'Avocado': 3, 'Banana': 4, 'Blueberry': 5, 'Cactus fruit': 6, 'Cantaloupe': 7, 'Cherry': 8, 'Clementine': 9, 'Corn': 10, 'Cucumber Ripe': 11, 'Grape Blue': 12, 'Kiwi': 13, 'Lemon': 14, 'Limes': 15, 'Mango': 16, 'Onion White': 17, 'Orange': 18, 'Papaya': 19, 'Passion Fruit': 20, 'Peach': 21, 'Pear': 22, 'Pepper Green': 23, 'Pepper Red': 24, 'Pineapple': 25, 'Plum': 26, 'Pomegranate': 27, 'Potato Red': 28, 'Raspberry': 29, 'Strawberry': 30, 'Tomato': 31, 'Watermelon': 32}


In [10]:
import joblib

joblib.dump(model, "model.pkl")

['model.pkl']